# E1 acoustic feature count comparison

Compare feature sets for Logistic Regression, SVM, and Random Forest.
Use balanced accuracy to compare results. Keep speakers separate across data splits.

1. Train each classifier with all 39 features as its baseline.
2. Compare feature counts on one training and validation split.
3. Select the count with the highest validation score above its baseline. If scores are equal, select fewer features. If no count improves the baseline, keep 39.
4. Check the selected counts on speaker folds. Fit the scaler, feature ranking, and model with each fold's training data only.
5. Train all three classifiers with their selected counts and all 39 features on training and validation data.
6. Compare each classifier's baseline and selected count on test data.

Start with a new kernel. Run the cells in order.
**Run All includes the test evaluation.**

**Limits:** The fold check is exploratory because validation data helped select the counts.
Data sources and prompts differ between classes. The scores cannot measure accent alone.


In [ ]:
# If required packages are missing, run the installation command. Then restart the kernel.
# %pip -q install "librosa>=0.10.2" "scikit-learn>=1.4" matplotlib pandas joblib soundfile

## 1. Settings and imports

Use the saved split and feature cache. If they are missing, create them.
Set `REBUILD_CACHE = True` to rebuild the features.
Use `FEATURE_COUNTS` to set the feature counts to test.

For each classifier, select the count with the highest validation score above its 39-feature baseline.
If scores are equal, select fewer features.
If no count improves the baseline, keep 39 features.


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Singlish')
else:
    PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'e1_feature_study'  # Folder for the saved split and feature cache.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = OUTPUT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
MAX_CLIPS_PER_CLASS = 1000  # Used only when no saved split exists.
REBUILD_CACHE = False  # Set True to rebuild features. Keep the saved speaker split.
CV_FOLDS = 3
FEATURE_COUNTS = [5, 10, 15, 20, 25, 30, 35, 39]
PERMUTATION_REPEATS = 5
BOOTSTRAP_REPEATS = 1000


SAMPLE_RATE = 16_000
MAX_SECONDS = 6
N_MFCC = 13
N_FFT = 1024
HOP_LENGTH = 256
EXTRACTION_VERSION = 'librosa39-v1'  # Extraction version used to check the saved cache.

print('Audio:', DATA_ROOT)


In [ ]:
import hashlib
import random
from collections import Counter
import json
from time import perf_counter
from itertools import combinations

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


def write_json(path, value):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + '\n')
    temporary.replace(path)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


print('Librosa:', librosa.__version__, '| scikit-learn:', sklearn.__version__)


MODEL_NAMES = ('logistic_regression', 'svm', 'random_forest')
MODEL_LABELS = {'logistic_regression': 'Logistic Regression', 'svm': 'SVM', 'random_forest': 'Random Forest'}


## 2. Prepare or load the speaker split and feature cache

Use the saved split to select clips and preserve speaker assignments.
If no split exists, select local audio and create a split with separate speakers in each set.
Audio files must be in `data/<class>/<speaker>/` folders.
This notebook does not download audio.

Load the saved 39-feature cache. If it is missing, extract and save the features.
Check clip IDs, labels, speaker separation, feature order, and finite values.
Also check audio file metadata and preprocessing settings.
If these settings or file metadata change, set `REBUILD_CACHE = True` to rebuild the features.
Keep the saved split to preserve the clip sample and speaker assignments.


In [ ]:
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
CLASS_LABELS = {'non_singapore': 0, 'singapore': 1}
SPLIT_NAMES = ('train', 'validation', 'test')
SPLIT_PATH = OUTPUT_DIR / 'speaker_splits.csv'


def clip_id(path):
    return path.relative_to(DATA_ROOT).as_posix()


def find_recordings(data_root):
    rows = []
    for class_name, label in CLASS_LABELS.items():
        class_dir = data_root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f'Missing class directory: {class_dir}')
        for path in sorted(class_dir.rglob('*')):
            if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
                continue
            parts = path.relative_to(class_dir).parts
            if len(parts) < 2:
                raise ValueError(f'Put this recording inside a speaker folder: {path}')
            rows.append((path, label, f'{class_name}::{parts[0]}'))
    if not rows:
        raise ValueError('No audio recordings found')
    return rows


def select_recordings(rows, limit, seed):
    if limit is None:
        return rows
    if not isinstance(limit, int) or isinstance(limit, bool) or limit < 1:
        raise ValueError('MAX_CLIPS_PER_CLASS must be None or a positive integer')
    rng = random.Random(seed)
    selected = []
    for label in CLASS_LABELS.values():
        by_speaker = {}
        for row in rows:
            if row[1] == label:
                by_speaker.setdefault(row[2], []).append(row)
        speakers = sorted(by_speaker)
        rng.shuffle(speakers)
        for speaker in speakers:
            rng.shuffle(by_speaker[speaker])
        count = 0
        while count < limit:
            added = False
            for speaker in speakers:
                if by_speaker[speaker]:
                    selected.append(by_speaker[speaker].pop())
                    count += 1
                    added = True
                    if count == limit:
                        break
            if not added:
                break
    return sorted(selected, key=lambda row: clip_id(row[0]))


def allocate_speakers(rows, seed):
    rng = np.random.default_rng(seed)
    assignment = {}
    for label in CLASS_LABELS.values():
        counts = Counter(group for _, item_label, group in rows if item_label == label)
        if len(counts) < 3:
            raise ValueError('Each class needs at least three speakers for three sets')
        bands = {}
        for group, count in sorted(counts.items()):
            bands.setdefault(count, []).append(group)
        for band in bands.values():
            rng.shuffle(band)
        n_test = max(1, round(len(counts) * 0.30))
        n_validation = max(1, round(len(counts) * 0.20))
        if n_test + n_validation >= len(counts):
            n_test = n_validation = 1

        def take_from_bands(number, split):
            total = sum(len(band) for band in bands.values())
            exact = {count: number * len(band) / total for count, band in bands.items()}
            quotas = {count: int(np.floor(value)) for count, value in exact.items()}
            remaining = number - sum(quotas.values())
            order = sorted(bands, key=lambda count: (-(exact[count] - quotas[count]), count))
            for count in order[:remaining]:
                quotas[count] += 1
            for count, band in bands.items():
                quota = quotas[count]
                for group in band[:quota]:
                    assignment[group] = split
                bands[count] = band[quota:]

        take_from_bands(n_test, 'test')
        take_from_bands(n_validation, 'validation')
        for band in bands.values():
            for group in band:
                assignment[group] = 'train'
    return assignment


def validate_split(clips):
    if clips.index.has_duplicates:
        raise ValueError('Duplicate clip IDs in the saved split')
    for row in clips.itertuples():
        parts = Path(row.Index).parts
        if len(parts) < 3 or parts[0] not in CLASS_LABELS or '..' in parts:
            raise ValueError(f'Invalid saved clip path: {row.Index}')
        if row.label != CLASS_LABELS[parts[0]] or row.speaker_group != f'{parts[0]}::{parts[1]}':
            raise ValueError(f'Invalid split metadata: {row.Index}')
        if not row.path.is_file():
            raise FileNotFoundError(row.path)
    if not set(clips['split']) <= set(SPLIT_NAMES):
        raise ValueError('Invalid split name')
    if (clips.groupby('speaker_group')['split'].nunique() > 1).any():
        raise ValueError('Speaker overlap between splits')
    for name in SPLIT_NAMES:
        if set(clips.loc[clips['split'] == name, 'label']) != {0, 1}:
            raise ValueError(f'Both labels must occur in {name}')


# Keep clip metadata in one table. Use the saved sample when a split exists.
if SPLIT_PATH.exists():
    clips = pd.read_csv(SPLIT_PATH, index_col='clip_id',
                        dtype={'clip_id': str, 'speaker_group': str, 'split': str})
    clips['path'] = [DATA_ROOT / identity for identity in clips.index]
    validate_split(clips)
else:
    recordings = select_recordings(find_recordings(DATA_ROOT), MAX_CLIPS_PER_CLASS, RANDOM_SEED)
    allocation = allocate_speakers(recordings, RANDOM_SEED)
    clips = pd.DataFrame([{
        'clip_id': clip_id(path), 'path': path, 'label': label,
        'speaker_group': group, 'split': allocation[group]
    } for path, label, group in recordings]).set_index('clip_id')
    validate_split(clips)
    clips[['label', 'speaker_group', 'split']].to_csv(SPLIT_PATH)
    del recordings, allocation
clips = clips.sort_index()
DATASET_ID = digest([(row.Index, int(row.label), row.speaker_group) for row in clips.itertuples()])
print('Saved split:', SPLIT_PATH)
print('Selected clips:', len(clips))


In [ ]:
# Prepare audio and extract features only when the cache must be built.
FEATURE_NAMES = (
    [f'mfcc_{i + 1}_mean' for i in range(N_MFCC)]
    + [f'mfcc_{i + 1}_std' for i in range(N_MFCC)]
    + ['f0_mean', 'f0_std', 'voiced_ratio']
    + [
        f'{family}_{stat}'
        for family in ('spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff', 'zcr', 'rms')
        for stat in ('mean', 'std')
    ]
)
assert len(FEATURE_NAMES) == 39


def load_and_prepare_audio(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)
    if y.size == 0:
        raise ValueError(f'No audible signal: {path}')
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    y = librosa.util.fix_length(y, size=max(len(y), SAMPLE_RATE))
    return y[:SAMPLE_RATE * MAX_SECONDS].astype(np.float32)


def mean_and_std(feature):
    return np.concatenate([np.mean(feature, axis=1), np.std(feature, axis=1)])


def extract_features(path):
    y = load_and_prepare_audio(path)
    mfcc = librosa.feature.mfcc(
        y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
    )
    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)
    f0 = librosa.yin(
        y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'),
        sr=SAMPLE_RATE, frame_length=2048, hop_length=HOP_LENGTH,
    )
    energy = rms.ravel()[:len(f0)]
    voiced = np.isfinite(f0) & (energy > max(float(np.median(energy)) * 0.1, 1e-4))
    voiced_f0 = f0[voiced]
    pitch_stats = [
        float(np.mean(voiced_f0)) if voiced_f0.size else 0.0,
        float(np.std(voiced_f0)) if voiced_f0.size else 0.0,
        float(np.mean(voiced)),
    ]
    time_features = [
        librosa.feature.spectral_centroid(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.spectral_bandwidth(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.spectral_rolloff(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH),
        rms,
    ]
    vector = np.concatenate([mean_and_std(mfcc), pitch_stats] + [mean_and_std(x) for x in time_features])
    vector = np.nan_to_num(vector, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    assert vector.shape == (len(FEATURE_NAMES),)
    return vector

CACHE_PATH = OUTPUT_DIR / 'acoustic_features_39.npz'
FAILURES_PATH = OUTPUT_DIR / 'extraction_failures.json'
CACHE_METADATA = {
    'dataset_id': DATASET_ID,
    'files': [],
    'extraction_version': EXTRACTION_VERSION,
    'settings': {'sample_rate': SAMPLE_RATE, 'max_seconds': MAX_SECONDS, 'n_mfcc': N_MFCC,
                 'n_fft': N_FFT, 'hop_length': HOP_LENGTH, 'trim_top_db': 30,
                 'normalization': 'peak', 'minimum_padded_seconds': 1,
                 'pitch_method': 'yin', 'pitch_frame_length': 2048, 'pitch_range': ['C2', 'C7']},
    'feature_names': FEATURE_NAMES,
    'librosa_version': librosa.__version__,
    'numpy_version': np.__version__,
}
for row in clips.itertuples():
    file_stat = row.path.stat()
    CACHE_METADATA['files'].append({
        'clip_id': row.Index, 'size': file_stat.st_size, 'mtime_ns': file_stat.st_mtime_ns})
CACHE_ID = digest(CACHE_METADATA)

if CACHE_PATH.exists() and not REBUILD_CACHE:
    with np.load(CACHE_PATH, allow_pickle=False) as cached:
        if str(cached['cache_id'].item()) != CACHE_ID:
            raise ValueError('Cache metadata changed. Set REBUILD_CACHE=True to rebuild features while preserving the split.')
        if cached['feature_names'].tolist() != FEATURE_NAMES:
            raise ValueError('Feature order in the cache does not match this extractor')
        X = cached['features'].copy()
        y = cached['labels'].copy()
        groups = cached['groups'].copy()
        clip_ids = cached['clip_ids'].copy()
        extraction_seconds = float(cached['extraction_seconds'].item())
        failures = json.loads(str(cached['failures_json'].item()))
    if not FAILURES_PATH.exists():
        write_json(FAILURES_PATH, failures)
    print('Loaded cache:', CACHE_PATH)
else:
    started = perf_counter()
    feature_rows, kept_ids, failures = [], [], []
    for number, row in enumerate(clips.itertuples(), start=1):
        try:
            vector = extract_features(row.path)
            if vector.shape != (len(FEATURE_NAMES),) or not np.isfinite(vector).all():
                raise ValueError('Invalid feature vector')
            feature_rows.append(vector)
            kept_ids.append(row.Index)
        except Exception as error:
            failures.append({'clip_id': row.Index, 'error': str(error)})
        if number % 100 == 0 or number == len(clips):
            print(f'Processed {number}/{len(clips)}; failures: {len(failures)}')
    if not feature_rows:
        raise ValueError('No clips survived feature extraction')
    X = np.vstack(feature_rows)
    clip_ids = np.asarray(kept_ids)
    y = clips.loc[clip_ids, 'label'].to_numpy(dtype=np.int64)
    groups = clips.loc[clip_ids, 'speaker_group'].to_numpy(dtype=str)
    extraction_seconds = perf_counter() - started
    np.savez_compressed(CACHE_PATH, features=X, labels=y, groups=groups, clip_ids=clip_ids,
                        feature_names=np.asarray(FEATURE_NAMES), cache_id=np.asarray(CACHE_ID),
                        extraction_seconds=np.asarray(extraction_seconds),
                        failures_json=np.asarray(json.dumps(failures)))
    write_json(FAILURES_PATH, failures)
    print('Saved cache:', CACHE_PATH)

if X.shape != (len(clip_ids), len(FEATURE_NAMES)) or not np.isfinite(X).all():
    raise ValueError('Invalid feature matrix')
if len(set(clip_ids)) != len(clip_ids) or len(y) != len(clip_ids) or len(groups) != len(clip_ids):
    raise ValueError('Invalid cache row identities')
if not set(clip_ids) <= set(clips.index):
    raise ValueError('Cache contains clips that are not in the saved split')
kept_clips = clips.loc[clip_ids]
if not np.array_equal(y, kept_clips['label'].to_numpy()) or not np.array_equal(groups, kept_clips['speaker_group'].to_numpy()):
    raise ValueError('Cache labels or speaker IDs differ from the saved split')
indices = {name: np.flatnonzero(kept_clips['split'].to_numpy() == name) for name in SPLIT_NAMES}
# Extraction failures must not remove either class from a split.
for name, rows in indices.items():
    if set(y[rows]) != {0, 1}:
        raise ValueError(f'Both labels must remain in {name} after extraction')
train_idx, val_idx, test_idx = (indices[name] for name in SPLIT_NAMES)
print('Features:', X.shape, '| initial extraction seconds:', round(extraction_seconds, 1))
print('Exclusions:', len(failures))
display(kept_clips.groupby(['split', 'label']).agg(
    clips=('path', 'size'), speakers=('speaker_group', 'nunique')))


## 3. Scoring and feature ranking

Fit the scaler and feature ranking with training speakers only.
Use balanced accuracy to compare models and feature counts.


In [ ]:
def make_model(name):
    if name == 'logistic_regression':
        estimator = Pipeline([('scale', StandardScaler()), ('classifier', LogisticRegression(
            C=1.0, class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED))])
    elif name == 'svm':
        estimator = Pipeline([('scale', StandardScaler()), ('classifier', SVC(
            kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_SEED,
            class_weight='balanced'))])
    elif name == 'random_forest':
        estimator = RandomForestClassifier(n_estimators=300, max_features='sqrt', min_samples_leaf=2,
            class_weight='balanced', bootstrap=True, n_jobs=-1, random_state=RANDOM_SEED)
    else:
        raise ValueError(f'Unknown classifier: {name}')
    return estimator


def score_predictions(rows):
    frame = pd.DataFrame(rows)
    truth, prediction = frame['true_label'], frame['prediction']
    return {
        'balanced_accuracy': float(balanced_accuracy_score(truth, prediction)),
        'macro_f1': float(f1_score(truth, prediction, average='macro', zero_division=0)),
    }


def predict_rows(estimator, columns, row_indices, **metadata):
    started = perf_counter()
    predictions = estimator.predict(X[row_indices][:, columns])
    seconds = perf_counter() - started
    rows = [{**metadata, 'clip_id': str(clip_ids[i]), 'speaker_group': str(groups[i]),
             'true_label': int(y[i]), 'prediction': int(predicted)}
            for i, predicted in zip(row_indices, predictions)]
    return rows, seconds


def fit_model(name, columns, row_indices):
    started = perf_counter()
    estimator = make_model(name)
    estimator.fit(X[row_indices][:, columns], y[row_indices])
    return estimator, perf_counter() - started


def rank_features(estimator, fit_idx):
    started = perf_counter()
    result = permutation_importance(estimator, X[fit_idx], y[fit_idx], scoring='balanced_accuracy',
        n_repeats=PERMUTATION_REPEATS, random_state=RANDOM_SEED, n_jobs=1)
    order = np.asarray(sorted(range(len(FEATURE_NAMES)),
        key=lambda i: (-float(result.importances_mean[i]), FEATURE_NAMES[i])), dtype=int)
    return order, result, perf_counter() - started


def paired_interval(candidate_rows, reference_rows, repeats=None, seed=None):
    # Keep clips from the same speaker together when estimating uncertainty.
    candidate = pd.DataFrame(candidate_rows).set_index('clip_id', verify_integrity=True)
    reference = pd.DataFrame(reference_rows).set_index('clip_id', verify_integrity=True)
    if set(candidate.index) != set(reference.index):
        raise ValueError('Paired comparison requires identical clips')
    reference = reference.loc[candidate.index]
    if not candidate[['speaker_group', 'true_label']].equals(reference[['speaker_group', 'true_label']]):
        raise ValueError('Paired comparison metadata mismatch')
    difference = (candidate.prediction == candidate.true_label).astype(float) - (
        reference.prediction == reference.true_label).astype(float)
    frame = candidate[['speaker_group', 'true_label']].assign(difference=difference)
    if (frame.groupby('speaker_group').true_label.nunique() != 1).any():
        raise ValueError('A speaker has mixed class labels')
    speakers = frame.groupby('speaker_group').agg(
        label=('true_label', 'first'), difference_sum=('difference', 'sum'), clips=('difference', 'size'))
    repeats = BOOTSTRAP_REPEATS if repeats is None else repeats
    rng = np.random.default_rng(RANDOM_SEED if seed is None else seed)
    samples = np.zeros(repeats)
    estimate = 0.0
    for label in (0, 1):
        class_speakers = speakers.loc[speakers.label == label]
        values = class_speakers['difference_sum'].to_numpy()
        clip_counts = class_speakers['clips'].to_numpy()
        if not len(values):
            raise ValueError('Both classes need speakers for paired intervals')
        draws = rng.integers(0, len(values), size=(repeats, len(values)))
        # Resample whole speakers. Give each clip equal influence within its class.
        samples += 0.5 * values[draws].sum(axis=1) / clip_counts[draws].sum(axis=1)
        estimate += 0.5 * values.sum() / clip_counts.sum()
    low, high = np.quantile(samples, [0.025, 0.975])
    return {'difference_from_full': float(estimate), 'interval_low': float(low), 'interval_high': float(high)}


def checked_counts(values):
    values = list(values)
    if not values or any(type(k) is not int or not 1 <= k <= len(FEATURE_NAMES) for k in values):
        raise ValueError('Feature counts must be integers from 1 to 39')
    return sorted(set(values) | {len(FEATURE_NAMES)})


def select_beating_count(rows, reference_score):
    eligible = [row for row in rows if row['balanced_accuracy'] > reference_score]
    if eligible:
        return min(eligible, key=lambda row: (-row['balanced_accuracy'], row['feature_count']))
    reference = next((row for row in rows if row['feature_count'] == len(FEATURE_NAMES)
                      and row['balanced_accuracy'] == reference_score), None)
    if reference is None:
        raise ValueError('No count beats baseline and no valid 39-feature fallback is available')
    return reference


if type(CV_FOLDS) is not int or CV_FOLDS < 2:
    raise ValueError('CV_FOLDS must be an integer of at least two')
if any(type(n) is not int or n < 1 for n in (PERMUTATION_REPEATS, BOOTSTRAP_REPEATS)):
    raise ValueError('Permutation and bootstrap repeats must be positive integers')
full_columns = np.arange(len(FEATURE_NAMES))
counts = checked_counts(FEATURE_COUNTS)
development_idx = np.concatenate([train_idx, val_idx])


def plot_validation_matrices(matrices, feature_counts, title):
    fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(13, 4))
    # Use the same color scale for all models. Each entry gives a clip count.
    maximum = max(sum(row) for matrix in matrices.values() for row in matrix)
    for name, axis in zip(MODEL_NAMES, axes):
        ConfusionMatrixDisplay(np.asarray(matrices[name]), display_labels=['Other', 'Singapore']).plot(
            ax=axis, colorbar=False, cmap='Blues', im_kw={'vmin': 0, 'vmax': maximum})
        axis.set_title(f"{MODEL_LABELS[name]} | {feature_counts[name]} features")
    fig.suptitle(title + ' — validation clip counts')
    fig.tight_layout()
    plt.show()

In [ ]:
def development_folds(split_groups):
    splitter = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    folds = []
    for fit_local, held_local in splitter.split(X[development_idx], y[development_idx], split_groups):
        fit_idx, held_idx = development_idx[fit_local], development_idx[held_local]
        if set(split_groups[fit_local]) & set(split_groups[held_local]):
            raise ValueError('Split-group overlap')
        if set(groups[fit_idx]) & set(groups[held_idx]):
            raise ValueError('Speaker overlap')
        if set(y[fit_idx]) != {0, 1} or set(y[held_idx]) != {0, 1}:
            raise ValueError('Each fold side must contain both labels')
        if np.intersect1d(np.concatenate([fit_idx, held_idx]), test_idx).size:
            raise ValueError('Test rows entered development')
        folds.append((fit_idx, held_idx))
    held = np.concatenate([rows for _, rows in folds])
    if len(held) != len(development_idx) or len(set(held)) != len(held) or set(held) != set(development_idx):
        raise ValueError('Each development clip must be held out exactly once')
    return folds


## 4. Train and validate the three 39-feature baselines

The first cell trains each classifier and the dummy reference with `train_idx` only.
The next cell makes predictions on `val_idx`.
It reports baseline scores and shows validation confusion matrices.


In [ ]:
# Train the 39-feature baselines with training speakers only.
baseline_models, baseline_fit_seconds = {}, {}
for name in MODEL_NAMES:
    baseline_models[name], baseline_fit_seconds[name] = fit_model(name, full_columns, train_idx)
dummy = DummyClassifier(strategy='most_frequent').fit(X[train_idx], y[train_idx])
print('Trained 39-feature baselines:', ', '.join(MODEL_LABELS[name] for name in MODEL_NAMES))

### Validate the 39-feature baselines


In [ ]:
# Evaluate the fitted 39-feature baselines on validation data.
validation_records, baseline_matrices, baseline_scores = {}, {}, []
for name in MODEL_NAMES:
    rows, predict_seconds = predict_rows(baseline_models[name], full_columns, val_idx,
        split='validation', model=name, feature_count=len(FEATURE_NAMES))
    metrics = {'model': name, 'feature_count': len(FEATURE_NAMES), **score_predictions(rows),
        'fit_seconds': baseline_fit_seconds[name], 'predict_seconds': predict_seconds, 'difference_from_full': 0.0}
    validation_records[(name, len(FEATURE_NAMES))] = {'metrics': metrics, 'predictions': rows,
                                                   'columns': full_columns.tolist()}
    baseline_scores.append(metrics)
    baseline_matrices[name] = confusion_matrix([row['true_label'] for row in rows],
        [row['prediction'] for row in rows], labels=[0, 1]).tolist()
dummy_rows, _ = predict_rows(dummy, full_columns, val_idx, split='validation')
display(pd.DataFrame(baseline_scores)[['model', 'feature_count', 'balanced_accuracy', 'macro_f1']].round(4))
print('Dummy validation balanced accuracy:', score_predictions(dummy_rows)['balanced_accuracy'])
plot_validation_matrices(baseline_matrices, {name: len(FEATURE_NAMES) for name in MODEL_NAMES},
    '39-feature baselines')

## 5. Compare feature counts on one split

The training cell ranks each baseline's features once. It uses training speakers only.
For each smaller count, train a model with the highest-ranked features.
Reuse the existing 39-feature model.

The next cell evaluates every count on the same validation speakers.
Keep the classifier settings fixed.
Save all results, including lower scores.
For each classifier, select the highest validation score that exceeds its 39-feature baseline score.
If scores are equal, select fewer features.
If no score exceeds the baseline, keep 39 features as a fallback.
This selection is provisional. It does not identify a global optimum or the minimum feature count.

Show each selected model's confusion matrix with its saved validation predictions.
This step does not need more training or a test evaluation.


In [ ]:
# Rank features with training speakers. Then train each smaller feature set.
count_models, training_importance = {}, {}
for name in MODEL_NAMES:
    ranking, importance, _ = rank_features(baseline_models[name], train_idx)
    training_importance[name] = (ranking, importance)
    for count in counts:
        columns = ranking[:count] if count < len(FEATURE_NAMES) else full_columns
        estimator, seconds = fit_model(name, columns, train_idx) if count < len(FEATURE_NAMES) else (
            baseline_models[name], baseline_fit_seconds[name])
        count_models[(name, count)] = {'model': estimator, 'columns': columns, 'fit_seconds': seconds}
print('Trained feature counts:', counts)

### Validate the feature counts and select candidates


In [ ]:
# Make validation predictions with the fitted models. Then select feature counts.
curve_scores, selected_counts_by_model, selected_count_rows = [], {}, []
for name in MODEL_NAMES:
    reference_score = validation_records[(name, len(FEATURE_NAMES))]['metrics']['balanced_accuracy']
    for count in counts:
        if count < len(FEATURE_NAMES):
            fitted = count_models[(name, count)]
            columns = fitted['columns']
            estimator, seconds = fitted['model'], fitted['fit_seconds']
            rows, predict_seconds = predict_rows(estimator, columns, val_idx,
                split='validation', model=name, feature_count=count)
            validation_records[(name, count)] = {'metrics': {'model': name, 'feature_count': count,
                **score_predictions(rows), 'fit_seconds': seconds, 'predict_seconds': predict_seconds},
                'predictions': rows, 'columns': columns.tolist()}
        record = validation_records[(name, count)]
        delta = record['metrics']['balanced_accuracy'] - reference_score
        qualifies = record['metrics']['balanced_accuracy'] > reference_score
        record['metrics'].update(difference_from_full=delta, beats_baseline=qualifies,
            finding='full reference' if count == len(FEATURE_NAMES) else
                    'beats baseline' if qualifies else 'equals baseline' if delta == 0 else 'below baseline')
        curve_scores.append(record['metrics'])
    model_rows = [row for row in curve_scores if row['model'] == name]
    selected = select_beating_count(model_rows, reference_score)
    selected_counts_by_model[name] = selected['feature_count']
    selection_status = 'best beating count' if selected['beats_baseline'] else 'baseline fallback: no tested count beats baseline'
    for row in model_rows:
        row['selected_for_folds'] = row['feature_count'] == selected['feature_count']
        if row['selected_for_folds'] and not selected['beats_baseline']:
            row['finding'] = selection_status
    selected_count_rows.append({**selected, 'reference_score': reference_score,
        'score_must_exceed': reference_score, 'selection_status': selection_status,
        'selection_rule': 'Highest validation score above the 39-feature baseline; ties prefer fewer features; otherwise keep 39.'})
display(pd.DataFrame(curve_scores)[['model', 'feature_count', 'balanced_accuracy', 'macro_f1',
    'difference_from_full', 'finding']].round(4))
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(15, 4.5), sharey=True)
for name, axis in zip(MODEL_NAMES, axes):
    rows = sorted([row for row in curve_scores if row['model'] == name], key=lambda row: row['feature_count'])
    selected = next(row for row in selected_count_rows if row['model'] == name)
    reference = next(row for row in rows if row['feature_count'] == len(FEATURE_NAMES))
    axis.plot([row['feature_count'] for row in rows], [100 * row['balanced_accuracy'] for row in rows],
              'o-', color='steelblue', label='Validation score')
    axis.axhline(100 * reference['balanced_accuracy'], color='black', linestyle='--', label='39-feature baseline')
    axis.scatter([selected['feature_count']], [100 * selected['balanced_accuracy']], s=140,
                 facecolors='none', edgecolors='darkorange', linewidths=2, label='Selected count (fallback if no improvement)')
    annotation = f"Best: {selected['feature_count']}" if selected['beats_baseline'] else f"Fallback: {selected['feature_count']}"
    axis.annotate(annotation, (selected['feature_count'], 100 * selected['balanced_accuracy']),
                  xytext=(0, 10), textcoords='offset points', ha='center', fontsize=9)
    axis.set(title=MODEL_LABELS[name], xlabel='Feature count', ylabel='Validation balanced accuracy (%)', xticks=counts)
    axis.grid(alpha=0.2)
    axis.margins(y=0.15)
axes[0].legend(fontsize=8)
fig.suptitle('Feature-count performance: saved train/validation split, no cross-validation')
fig.tight_layout()
plt.show()
display(pd.DataFrame(selected_count_rows)[['model', 'feature_count', 'balanced_accuracy',
    'difference_from_full', 'selection_status']].round(4))
selected_matrices = {name: confusion_matrix(
    [row['true_label'] for row in validation_records[(name, count)]['predictions']],
    [row['prediction'] for row in validation_records[(name, count)]['predictions']], labels=[0, 1]).tolist()
    for name, count in selected_counts_by_model.items()}
plot_validation_matrices(selected_matrices, selected_counts_by_model,
    'Selected feature-count candidates')

### Training feature importance

Shuffle one feature at a time and measure the decrease in training balanced accuracy.
A larger decrease means the model relies more on that feature.
Blue bars mark the top features for the validation-selected count. Gray bars mark the remaining features.
Lines show the variation across shuffle repeats.


In [ ]:
# Show the training permutation importance used to rank features for each classifier.
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(18, 11))
for name, axis in zip(MODEL_NAMES, axes):
    ranking, importance = training_importance[name]
    count = selected_counts_by_model[name]
    order = ranking[::-1]
    selected_features = set(ranking[:count])
    colors = ['steelblue' if i in selected_features else 'lightgray' for i in order]
    axis.barh([FEATURE_NAMES[i] for i in order], 100 * importance.importances_mean[order],
              xerr=100 * importance.importances_std[order], color=colors, error_kw={'elinewidth': 0.7})
    axis.axvline(0, color='black', linewidth=0.7)
    axis.set(title=f'{MODEL_LABELS[name]}: {count} selected features',
             xlabel='Training balanced accuracy decrease (percentage points)')
    axis.tick_params(axis='y', labelsize=8)
fig.suptitle('Training feature importance: blue = top features for the selected count')
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()


## 6. Check selected counts on speaker folds and report results

Create the development folds after the validation step.
Keep each classifier's selected count fixed.
Evaluate that count and the 39-feature reference for each classifier.

Fit the feature ranking and scaler again with each fold's training speakers only.
Do not reuse the feature names from the single-split ranking.
Some speakers held out in the folds contributed to that ranking.
If the selected count is 39, reuse the full model. Do not train a duplicate model.

Compare combined scores, individual fold scores, and feature-name agreement.
The check measures performance and feature-name agreement at the selected count.
Development labels helped select these counts.
Thus, the fold check does not independently evaluate count selection.
It does not measure how often a new sample would select the same count.

Show tested counts, score decreases, fold variation, and feature-name agreement in the notebook.

The chart compares selected features with the 39-feature baseline.
Bars show balanced accuracy from all held-out fold predictions combined.
Dots show balanced accuracy for each individual fold.
The combined score is not necessarily the average of the individual fold scores.


In [ ]:
def run_folds(name, selected_count, split_folds):
    records = {}
    for fold, (fit_idx, held_idx) in enumerate(split_folds, start=1):
        full, full_seconds = fit_model(name, full_columns, fit_idx)
        ranking = full_columns
        if selected_count < len(FEATURE_NAMES):
            ranking, _, _ = rank_features(full, fit_idx)
        for count in sorted({selected_count, len(FEATURE_NAMES)}):
            columns = full_columns if count == len(FEATURE_NAMES) else ranking[:count]
            estimator, seconds = (full, full_seconds) if count == len(FEATURE_NAMES) else fit_model(name, columns, fit_idx)
            predictions, predict_seconds = predict_rows(estimator, columns, held_idx,
                model=name, feature_count=count, fold=fold)
            records[(name, count, fold)] = {'metrics': {'model': name, 'feature_count': count,
                'fold': fold, **score_predictions(predictions), 'fit_seconds': seconds,
                'predict_seconds': predict_seconds}, 'predictions': predictions, 'columns': columns.tolist()}
    return records


def summarize_folds(records):
    summary = []
    for name, count in sorted({(name, count) for name, count, _ in records}):
        entries = [records[(name, count, fold)] for fold in range(1, CV_FOLDS + 1)]
        references = [records[(name, len(FEATURE_NAMES), fold)] for fold in range(1, CV_FOLDS + 1)]
        rows = [row for entry in entries for row in entry['predictions']]
        reference_rows = [row for entry in references for row in entry['predictions']]
        sets = [set(entry['columns']) for entry in entries]
        scores = [entry['metrics']['balanced_accuracy'] for entry in entries]
        summary.append({'model': name, 'feature_count': count,
            **score_predictions(rows), 'mean_fold_score': float(np.mean(scores)),
            'fold_score_std': float(np.std(scores, ddof=1)), 'min_fold_score': float(min(scores)),
            'max_fold_score': float(max(scores)), 'mean_feature_jaccard': float(np.mean([
                len(left & right) / len(left | right) for left, right in combinations(sets, 2)])),
            'folds_matching_baseline': sum(score >= reference['metrics']['balanced_accuracy']
                for score, reference in zip(scores, references)), **paired_interval(rows, reference_rows)})
    return summary


folds = development_folds(groups[development_idx])
fold_table = pd.DataFrame([{'fold': fold, 'role': role, 'clip_id': clip_ids[i],
    'speaker_group': groups[i], 'label': int(y[i])}
    for fold, (fit_idx, held_idx) in enumerate(folds, start=1)
    for role, rows in (('fit', fit_idx), ('held_out', held_idx)) for i in rows])
display(fold_table.groupby(['fold', 'role', 'label']).agg(clips=('clip_id', 'size'), speakers=('speaker_group', 'nunique')))
cv_records = {}
for name in MODEL_NAMES:
    records = run_folds(name, selected_counts_by_model[name], folds)
    cv_records.update(records)
fold_summary = summarize_folds(cv_records)
display(pd.DataFrame(fold_summary)[['model', 'feature_count', 'balanced_accuracy',
    'min_fold_score', 'max_fold_score', 'difference_from_full', 'folds_matching_baseline', 'mean_feature_jaccard']].round(4))

In [ ]:
journey_rows = []
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(15, 4.5), sharey=True)
for name, axis in zip(MODEL_NAMES, axes):
    selected_count = selected_counts_by_model[name]
    selected = next(row for row in fold_summary if row['model'] == name and row['feature_count'] == selected_count)
    reference = next(row for row in fold_summary if row['model'] == name and row['feature_count'] == len(FEATURE_NAMES))
    entries = [reference] if selected_count == len(FEATURE_NAMES) else [selected, reference]
    labels = ['39 (selected/full)'] if len(entries) == 1 else [f'{selected_count} (selected)', '39 (full)']
    axis.bar(labels, [100 * row['balanced_accuracy'] for row in entries], color=['steelblue', 'gray'][:len(entries)])
    for j, row in enumerate(entries):
        fold_scores = [100 * cv_records[(name, row['feature_count'], fold)]['metrics']['balanced_accuracy']
                       for fold in range(1, CV_FOLDS + 1)]
        axis.scatter(j + np.linspace(-0.12, 0.12, CV_FOLDS), fold_scores, color='black', s=20, zorder=3)
    axis.set(title=MODEL_LABELS[name], ylabel='Development balanced accuracy (%)', ylim=(0, 100))
    fold_finding = ('full set selected' if selected_count == len(FEATURE_NAMES) else
        'below pooled baseline' if selected['difference_from_full'] < 0 else
        'mixed across folds' if selected['folds_matching_baseline'] < CV_FOLDS else 'promising across folds')
    for row in [r for r in curve_scores if r['model'] == name]:
        journey_rows.append({'stage': 'validation curve', 'model': name, 'feature_count': row['feature_count'],
            'balanced_accuracy': row['balanced_accuracy'],
            'difference_from_full': row['difference_from_full'], 'finding': row['finding'],
            'selected_for_folds': row['feature_count'] == selected_count})
    journey_rows.append({'stage': 'speaker-fold check', 'model': name, 'feature_count': selected_count,
        'balanced_accuracy': selected['balanced_accuracy'],
        'difference_from_full': selected['difference_from_full'], 'finding': fold_finding,
        'selected_for_folds': True, 'folds_matching_baseline': selected['folds_matching_baseline'],
        'mean_feature_jaccard': selected['mean_feature_jaccard'],
        'min_fold_score': selected['min_fold_score'], 'max_fold_score': selected['max_fold_score']})
fig.suptitle('Selected features vs 39-feature baseline: combined and individual fold scores')
fig.text(0.5, 0.01, 'Bars: balanced accuracy from all held-out predictions combined. Dots: balanced accuracy for each fold.',
         ha='center', fontsize=9)
fig.tight_layout(rect=(0, 0.06, 1, 0.94))
plt.show()
display(pd.DataFrame(journey_rows)[['stage', 'model', 'feature_count', 'balanced_accuracy', 'difference_from_full', 'finding']].round(4))

## 7. Train all classifiers on training and validation data

For each classifier, train the 39-feature baseline and its validation-selected count.
Rank features again with training and validation data only.
Keep each classifier's selected count fixed.
If the selected count is 39, reuse the baseline model.
Save the fitted models and feature names before the test.


In [ ]:
# Train each classifier's baseline and selected count. Do not use test data.
final_models = {}
for name in MODEL_NAMES:
    full, _ = fit_model(name, full_columns, development_idx)
    final_models[(name, len(FEATURE_NAMES))] = (full, full_columns)
    count = selected_counts_by_model[name]
    if count < len(FEATURE_NAMES):
        ranking, _, _ = rank_features(full, development_idx)
        columns = ranking[:count]
        reduced, _ = fit_model(name, columns, development_idx)
        final_models[(name, count)] = (reduced, columns)
    print(MODEL_LABELS[name], '| Selected count:', count)

for (name, count), (estimator, columns) in final_models.items():
    joblib.dump({'model': estimator, 'columns': columns.tolist(),
                 'feature_names': [FEATURE_NAMES[i] for i in columns]},
                RESULTS_DIR / f'{name}_{count}.joblib')
# Keep the results needed by Part 2 and the separate comparison notebooks.
joblib.dump({
    'selected_counts': selected_counts_by_model,
    'validation_predictions': [row for (name, count), record in validation_records.items()
        if count in (len(FEATURE_NAMES), selected_counts_by_model[name])
        for row in record['predictions']]
}, RESULTS_DIR / 'study_results.joblib')


## 8. Test all classifiers

Compare each classifier's 39-feature baseline and validation-selected count on the same test clips.
Use the fitted models from section 7. Do not change feature counts or settings from test results.
If the selected count is 39, both rows use the same model and predictions.
Repeated use of the same test speakers does not give new independent confirmation.


In [ ]:
# Evaluate all three classifiers. Keep their validation-selected counts fixed.
final_scores, test_predictions = [], {}
for name in MODEL_NAMES:
    for count in sorted({len(FEATURE_NAMES), selected_counts_by_model[name]}):
        estimator, columns = final_models[(name, count)]
        rows, _ = predict_rows(estimator, columns, test_idx, model=name, feature_count=count)
        test_predictions[(name, count)] = rows
    baseline_score = score_predictions(test_predictions[(name, len(FEATURE_NAMES))])['balanced_accuracy']
    for role, count in [('39-feature baseline', len(FEATURE_NAMES)),
                        ('Validation-selected count', selected_counts_by_model[name])]:
        metrics = score_predictions(test_predictions[(name, count)])
        final_scores.append({'model': name, 'role': role, 'feature_count': count,
            **metrics, 'difference_from_39': metrics['balanced_accuracy'] - baseline_score})

display(pd.DataFrame(final_scores).assign(model=lambda frame: frame.model.map(MODEL_LABELS)).round(4))
print('Test clips:', len(test_idx), '| Test speakers:', len(set(groups[test_idx])))
print('Repeated use of these test speakers does not give new independent confirmation.')
study_results = joblib.load(RESULTS_DIR / 'study_results.joblib')
study_results['test_scores'] = final_scores
joblib.dump(study_results, RESULTS_DIR / 'study_results.joblib')

fig, axes = plt.subplots(len(MODEL_NAMES), 3, figsize=(14, 11))
for name, row_axes in zip(MODEL_NAMES, axes):
    count = selected_counts_by_model[name]
    scores = [score for score in final_scores if score['model'] == name]
    bars = row_axes[0].bar(['39 baseline', f'{count} selected'],
                    [100 * score['balanced_accuracy'] for score in scores], color=['gray', 'steelblue'])
    row_axes[0].bar_label(bars, fmt='%.2f%%', padding=3, fontsize=9)
    row_axes[0].set(title=MODEL_LABELS[name], ylabel='Test balanced accuracy (%)', ylim=(0, 100))
    for axis, feature_count, label in zip(row_axes[1:], [len(FEATURE_NAMES), count], ['Baseline', 'Selected']):
        rows = test_predictions[(name, feature_count)]
        matrix = confusion_matrix([row['true_label'] for row in rows], [row['prediction'] for row in rows], labels=[0, 1])
        ConfusionMatrixDisplay(matrix, display_labels=['Other', 'Singapore']).plot(ax=axis, colorbar=False, cmap='Blues')
        axis.set_title(f'{MODEL_LABELS[name]}: {label} ({feature_count} features)')
fig.suptitle('Test results: 39-feature baselines and validation-selected counts')
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()


## Saved files

Tables and charts are shown in this notebook. Detailed report files are not exported.
The speaker split and feature cache stay in `outputs/e1_feature_study`.
The fitted models and shared results stay in `outputs/e1_feature_study/results`.
The shared results file is used by Part 2 and the separate comparison notebooks.
